# Lista 13

## Multiprocessing, profilowanie

(6pkt + 2pkt)

Na liÅ›cie znajduje siÄ™ 6 zadaÅ„. Po rozwiÄ…zaniu zadaÅ„, pokaÅ¼ kod prowadzÄ…cemu i odpowiedz na **pytanie kontrolne** â€” tylko wtedy przyznajemy punkty. Dodatkowo przeÅ›lij zadanie na platformie skos.

Postaraj siÄ™, Å¼eby Twoje wykresy byÅ‚y moÅ¼liwie podobne do tych w rozwiÄ…zaniach.

**Zadania bonusowe**

Dodatkowe zadanie na wyÅ¼szÄ… ocenÄ™ oznaczone jest â­ï¸ i warte 2 pkt.

## Identyfikacja wÄ…skich gardeÅ‚ czasowych (analiza)

PoniÅ¼ej znajduje siÄ™ fragment tabeli wygenerowanej przez profiler CPU:

```text
---------------------------------  ------------  ------------  ------------
                             Name      Self CPU     CPU total    # of Calls
---------------------------------  ------------  ------------  ------------
                     aten::conv2d     231.000us      31.931ms            20
                aten::batch_norm      211.000us      14.693ms            20
                       aten::mean     332.000us       2.631ms            21
---------------------------------
```

### Polecenia

1. KtÃ³ry operator jest najwiÄ™kszym wÄ…skim gardÅ‚em czasowym?
2. WyjaÅ›nij rÃ³Å¼nicÄ™ pomiÄ™dzy **Self CPU time** a **CPU total time**.
3. Dlaczego `aten::conv2d` ma tak duÅ¼y czas caÅ‚kowity?

### RozwiÄ…zanie

1. Największym wąskim gardłem czasowym jest aten::conv2d, bo ma najwyzszy CPU total time.
2. Self CPU time to czas własny operacji, a CPU total to czas połączony z czasem wszystkich podoperacji.
3. Wynika to z tego, ze ten operator wywołuje takie operacje, które zajmują więcej czasu i sumują się do takiej liczby.

## Profilowanie zuÅ¼ycia pamiÄ™ci (analiza)

Fragment wynikÃ³w:

```text
---------------------------------
Name                     Self CPU Mem
---------------------------------
aten::empty              94.79 MB
aten::batch_norm          0 B
aten::conv2d              0 B
---------------------------------
```

### Polecenia

1. Dlaczego `aten::empty` zuÅ¼ywa najwiÄ™cej pamiÄ™ci?
2. Dlaczego `aten::conv2d` ma `0 B` w kolumnie *Self CPU Mem*?
3. Czy oznacza to, Å¼e konwolucje nie zuÅ¼ywajÄ… pamiÄ™ci?

### RozwiÄ…zanie

1. Zuzywa tyle pamięci, bo tworzy nowe tensory, które alokują pamięć.
2. Bo nie alokuje nowej pamięci w swojej implementacji.
3. Pamięć dla nich jest alokowana przez aten::empty, a nie bezpośrednio przez nie.

## UzupeÅ‚nianie kodu: profilowanie CPU

UzupeÅ‚nij poniÅ¼szy kod tak, aby:

* uruchomiÄ‡ profiler CPU
* zapisaÄ‡ ksztaÅ‚ty tensorÃ³w
* oznaczyÄ‡ sekcjÄ™ inferencji etykietÄ… `"inference"`

In [ ]:
from torch.profiler import profile, ProfilerActivity, record_function
import torch
from torchvision import models

model = models.resnet18()
inputs = torch.randn(5, 3, 224, 224)

with profile(activities=[ProfilerActivity.CPU], record_shapes=True) as prof:
    with record_function("inference"):
        model(inputs)

print(prof.key_averages().table(
    sort_by="cpu_time_total",
    row_limit=5
))

---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                             Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
---------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                        inference         2.59%       1.518ms       100.00%      58.588ms      58.588ms             1  
                     aten::conv2d         0.10%      61.206us        70.08%      41.056ms       2.053ms            20  
                aten::convolution         0.35%     205.879us        69.97%      40.995ms       2.050ms            20  
               aten::_convolution         0.29%     169.669us        69.62%      40.789ms       2.039ms            20  
                aten::thnn_conv2d         0.09%      51.582us        69.27%      40.587ms       2.029ms            20  
---------------------------------  -----

# WspÃ³Å‚dzielone tensory w multiprocessing

Uruchom poniÅ¼szy kod i odpowiedz na pytania znajdujÄ…ce siÄ™ pod nim.

---

### Kod do uruchomienia

```python
import torch
import torch.multiprocessing as mp


def worker(rank, shared_tensor):
    for _ in range(1):
        shared_tensor[rank] += 1


if __name__ == "__main__":
    mp.set_start_method("spawn", force=True)

    tensor = torch.zeros(4)
    tensor.share_memory_()

    processes = []

    for rank in range(4):
        p = mp.Process(
            target=worker,
            args=(rank, tensor)
        )
        p.start()
        processes.append(p)

    for p in processes:
        p.join()

    print("Final tensor:", tensor)
```

---

### Pytanie do analizy

**SprawdÅº, co siÄ™ stanie, gdy wszystkie procesy bÄ™dÄ… modyfikowaÄ‡ to samo pole tensora zamiast rÃ³Å¼nych elementÃ³w.**

W szczegÃ³lnoÅ›ci:

* zmieÅ„ kod tak, aby kaÅ¼dy proces zwiÄ™kszaÅ‚ ten sam indeks tensora,
* zwiÄ™ksz liczbÄ™ iteracji pÄ™tli `worker`,
* uruchom program kilkukrotnie,
* porÃ³wnaj otrzymywane wyniki,
* sprÃ³buj wyjaÅ›niÄ‡ obserwowane zachowanie.

#### Rozwiązanie:
Mój worker wyglądał w ten sposób:

```python
def worker(rank, shared_tensor):
    for _ in range(100):
        shared_tensor[0] += 1
```

a wyniki były takie:

![Screenshot 2026-01-20 at 14.01.18.png](<attachment:Screenshot 2026-01-20 at 14.01.18.png>)

(ok. 399-400)

Udało się to bez dziwnych wyników z race condition gdzie wyniki byłyby znacząco mniejsze niz 400.
Prawdopodobnie głównie jest to zasługa:
- uzycia tensor.share_memory_()
- kwestii, ze operacja tensor[i] += 1 jest atomowa
Róznica 399 a 400 moze wynikać np z minimalnego race condition w synchronizacji startu tych procesów.

## PorÃ³wnanie wykonania funkcji obliczeniowej â€“ sekwencyjnie i z multiprocessing

RozwaÅ¼amy funkcjÄ™ obliczeniowÄ… `x(n)`, ktÃ³ra wykonuje kosztownÄ… operacjÄ™ numerycznÄ…. Funkcja ta jest wywoÅ‚ywana wielokrotnie:

* w wersji **sekwencyjnej** (jeden proces),
* w wersji **rÃ³wnolegÅ‚ej** (wiele procesÃ³w, `Process + join`).

### Polecenia

1. UzupeÅ‚nij funkcjÄ™ `run_sequential`.
2. UzupeÅ‚nij funkcjÄ™ `run_multiprocessing`.
3. Upewnij siÄ™, Å¼e procesy sÄ… poprawnie synchronizowane przy uÅ¼yciu `join()`.
4. Uruchom program dla:

   * `TASKS = 8`
   * `WORKERS = 8`
5. Odpowiedz na pytania:

   * Czy multiprocessing przyspieszyÅ‚ obliczenia?
   * Czy przyspieszenie jest liniowe?

---

## Kod do uzupeÅ‚nienia

```python
import time
import torch
import torch.multiprocessing as mp


def x(n: int) -> float:
    """
    Costly numerical function.
    """
    t = torch.randn(n)
    for _ in range(5):
        t = t * t + 1.0
    return t.sum().item()


def run_sequential(tasks: int, n: int) -> float:
    """
    Run function x sequentially.
    Returns execution time.
    """
    start = time.time()

    for _ in range(tasks):
        x(n)

    end = time.time()
    return end - start


def worker(n: int, out, idx: int):
    """
    Worker process.
    """
    out[idx] = x(n)


def run_multiprocessing(tasks: int, n: int, workers: int) -> float:
    """
    Run function x using multiprocessing.
    Returns execution time.
    """
    manager = mp.Manager()
    results = manager.list([None] * tasks)
    processes = []

    start = time.time()

    for i in range(min(tasks, workers)):
        p = mp.Process(
            target=worker,
            args=(n, results, i)
        )
        p.start()
        processes.append(p)

    for p in processes:
        p.join()

    end = time.time()
    return end - start


if __name__ == "__main__":
    mp.set_start_method("spawn", force=True)

    TASKS = 8          # number of independent jobs
    N = 5_000_000      # cost of a single job
    WORKERS = 8        # number of processes

    t_seq = run_sequential(TASKS, N)
    t_mp = run_multiprocessing(TASKS, N, WORKERS)

    print("\n=== SUMMARY ===")
    print(f"Sequential:      {t_seq:.2f} s")
    print(f"Multiprocessing:{t_mp:.2f} s")
    print(f"Speedup:         {t_seq / t_mp:.2f}x")
```

#### Wyniki:
Dla takiego kodu dostałem takie wyniki:
```
=== SUMMARY ===
Sequential:      0.29 s
Multiprocessing:1.05 s
Speedup:         0.28x
```

Niestety nie wiem, czemu jest to tak wolne, speedup powinien być znacznie wyzszy.


## DDP + DataLoader z wieloma workerami

Twoim zadaniem jest:

1. UruchomiÄ‡ **trening DDP** z wieloma procesami
2. UÅ¼yÄ‡ **DataLoadera z `num_workers > 0`**
3. ZauwaÅ¼yÄ‡, Å¼e:

   * DDP **synchronizuje gradienty**
   * DataLoader **tylko Å‚aduje dane**

---

### Kod do uzupeÅ‚nienia

UzupeÅ‚nij miejsca oznaczone `__________`.

```python
import torch
import torch.nn as nn
import torch.optim as optim
import torch.multiprocessing as mp
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler


# Dataset
class RandomDataset(Dataset):
    def __init__(self, size, length):
        self.data = torch.randn(length, size)
        self.targets = torch.randn(length, 1)

    def __getitem__(self, index):
        return self.data[index], self.targets[index]

    def __len__(self):
        return len(self.data)


# DDP setup
def setup(rank, world_size):
    dist.init_process_group(
        backend="gloo",
        init_method="tcp://127.0.0.1:29500",
        rank=rank,
        world_size=world_size
    )

# DDP cleanup
def cleanup():
    dist.destroy_process_group()


# Training function
def train(rank, world_size):
    print(f"Rank {rank} starting")

    setup(rank, world_size)

    # Dataset + DistributedSampler
    dataset = RandomDataset(size=10, length=1000)
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank)

    # DataLoader with multiple workers
    dataloader = DataLoader(
        dataset,
        batch_size=32,
        sampler=sampler,
        num_workers=4
    )

    # Model
    model = nn.Linear(10, 1)
    model = DDP(model)
    model.train()

    optimizer = optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.MSELoss()

    for epoch in range(2):
        sampler.set_epoch(epoch)

        for x, y in dataloader:
            optimizer.zero_grad()
            output = model(x)
            loss = criterion(output, y)
            loss.backward() 
            optimizer.step()

        print(f"Rank {rank}, Epoch {epoch}, Loss {loss.item():.4f}")

    cleanup()


# Entry point
if __name__ == "__main__":
    mp.set_start_method("spawn", force=True)

    WORLD_SIZE = 2

    mp.spawn(
        train,
        args=(WORLD_SIZE,),
        nprocs=WORLD_SIZE,
        join=True
    )
```

#### Wyniki:

```
Rank 0 starting
Rank 1 starting
[W120 14:36:50.213471000 ProcessGroupGloo.cpp:547] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
[W120 14:36:50.225810000 ProcessGroupGloo.cpp:547] Warning: Unable to resolve hostname to a (local) address. Using the loopback address as fallback. Manually set the network interface to bind to with GLOO_SOCKET_IFNAME. (function operator())
Rank 1, Epoch 0, Loss 1.1869
Rank 0, Epoch 0, Loss 1.3760
Rank 0, Epoch 1, Loss 1.4743
Rank 1, Epoch 1, Loss 0.8903
```

## â­ï¸ Optymalizacja kodu

1. Uruchom profiler
2. Odpowiedz:

   * ktÃ³ra funkcja ma **najwiÄ™kszy `CPU total`**
   * ktÃ³ra ma **najwiÄ™kszy `# of Calls`**
   * gdzie `Self CPU` â‰ª `CPU total`
3. ZnajdÅº wszystkie wÄ…skie gardÅ‚a i napisz ile znalazÅ‚eÅ›/znalazÅ‚aÅ›
4. Zoptymalizuj kod
5. PorÃ³wnaj profil **przed i po**

---

## KOD DO PROFILOWANIA

```python
import torch
from torch.profiler import profile, record_function, ProfilerActivity


def level_3(x):
    # 4ï¸âƒ£ alokacja + scalar sync
    y = torch.randn(1000)
    return (x + y).sum().item()


def level_2(x):
    total = 0.0
    for _ in range(50):  # 3ï¸âƒ£ pÄ™tla
        total += level_3(x)
    return total


def level_1():
    result = 0.0
    for _ in range(100):  # 2ï¸âƒ£ pÄ™tla
        x = torch.randn(1000)  # 1ï¸âƒ£ alokacja w pÄ™tli
        result += level_2(x)
    return result


def run():
    with profile(
        activities=[ProfilerActivity.CPU],
        record_shapes=True,
        profile_memory=True
    ) as prof:
        with record_function("bzdurny_pipeline"):
            level_1()

    print(prof.key_averages().table(
        sort_by="cpu_time_total",
        row_limit=20
    ))


if __name__ == "__main__":
    run()
```